In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

train = pd.read_csv("../data/processed/train_feature_engineered.csv")
print("✔ Loaded —", train.shape)


✔ Loaded — (307511, 136)


In [11]:
y = train["TARGET"]
X = train.select_dtypes(include=[np.number]).drop(columns=["TARGET"])


In [12]:
# Replace infinities with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Remove columns that are entirely NaN
X = X.dropna(axis=1, how="all")

# Impute remaining NaN with median
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print("Remaining NaN:", np.isnan(X.values).sum())


Remaining NaN: 0


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("✔ Split OK")


✔ Split OK


In [14]:
log_reg = LogisticRegression(
    max_iter=5000,
    solver="saga"
)

In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train_scaled, y_train)

log_pred = log_reg.predict_proba(X_test_scaled)[:, 1]
log_auc = roc_auc_score(y_test, log_pred)

print(f"🔹 Logistic Regression AUC: {log_auc:.4f}")


🔹 Logistic Regression AUC: 0.7427


C:\Users\Tambo\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [16]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)

print(f"🔹 Random Forest AUC: {rf_auc:.4f}")


🔹 Random Forest AUC: 0.7423


In [17]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=300
)

hgb.fit(X_train, y_train)
hgb_pred = hgb.predict_proba(X_test)[:, 1]
hgb_auc = roc_auc_score(y_test, hgb_pred)

print(f"🔹 HistGradientBoosting AUC: {hgb_auc:.4f}")


🔹 HistGradientBoosting AUC: 0.7625


In [18]:
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3
)

gb.fit(X_train, y_train)
gb_pred = gb.predict_proba(X_test)[:, 1]
gb_auc = roc_auc_score(y_test, gb_pred)

print(f"🔹 Gradient Boosting AUC: {gb_auc:.4f}")



🔹 Gradient Boosting AUC: 0.7585


In [19]:
print("\n📊 Model Comparison (AUC)")
print(f"Logistic Regression:      {log_auc:.4f}")
print(f"Random Forest:            {rf_auc:.4f}")
print(f"HistGradientBoosting:     {hgb_auc:.4f}")
print(f"Gradient Boosting:        {gb_auc:.4f}")




📊 Model Comparison (AUC)
Logistic Regression:      0.7427
Random Forest:            0.7423
HistGradientBoosting:     0.7625
Gradient Boosting:        0.7585
